# Data Processing – INX Future Inc Employee Performance

**Purpose of this notebook:** load the original employee data, check its quality, clean it, and save the final datasets used for analysis and modelling.

**Steps**
1. Load the raw data and the data definitions sheet
2. Check the structure, missing values, duplicates and data types
3. Check for outliers and logical inconsistencies
4. Transform the columns (readable labels for reporting, numeric codes for modelling)
5. Save the processed datasets to `data/processed`

The raw file is never modified.

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

ROOT = Path("../..").resolve()
RAW_FILE = ROOT / "data" / "raw" / "INX_Future_Inc_Employee_Performance_CDS_Project2_Data_V1_8.xls"
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "src" / "models"
FIGURES = ROOT / "src" / "visualization" / "figures"
TARGET = "PerformanceRating"
RANDOM_STATE = 42

## 1. Load the raw data

In [2]:
data = pd.read_excel(RAW_FILE, engine="xlrd", sheet_name=0)
definitions = pd.read_excel(RAW_FILE, engine="xlrd", sheet_name=1, header=None)
print("Rows and columns:", data.shape)
data.head()

Rows and columns: (1200, 28)


,EmpNumber,Age,Gender,EducationBackground,MaritalStatus,EmpDepartment,EmpJobRole,BusinessTravelFrequency,DistanceFromHome,EmpEducationLevel,...,EmpRelationshipSatisfaction,TotalWorkExperienceInYears,TrainingTimesLastYear,EmpWorkLifeBalance,ExperienceYearsAtThisCompany,ExperienceYearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition,PerformanceRating
0,E1001000,32,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,10,3,...,4,10,2,2,10,7,0,8,No,3
1,E1001006,47,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,14,4,...,4,20,2,3,7,7,1,7,No,3
2,E1001007,40,Male,Life Sciences,Married,Sales,Sales Executive,Travel_Frequently,5,4,...,3,20,2,3,18,13,1,12,No,4
3,E1001009,41,Male,Human Resources,Divorced,Human Resources,Manager,Travel_Rarely,10,4,...,2,23,2,2,21,6,12,6,No,3
4,E1001010,60,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,16,4,...,4,10,1,3,2,2,2,2,No,3


## 2. Structure and quality checks

In [3]:
print("Missing values in total:", int(data.isna().sum().sum()))
print("Duplicate rows:", int(data.duplicated().sum()))
print("Unique employee numbers:", data["EmpNumber"].nunique(), "of", len(data))
data.dtypes.value_counts()

Missing values in total: 0
Duplicate rows: 0
Unique employee numbers: 1200 of 1200


int64    19
str       9
Name: count, dtype: int64

In [4]:
data.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Age,1200.0,36.92,9.09,18.0,30.0,36.0,43.0,60.0
DistanceFromHome,1200.0,9.17,8.18,1.0,2.0,7.0,14.0,29.0
EmpEducationLevel,1200.0,2.89,1.04,1.0,2.0,3.0,4.0,5.0
EmpEnvironmentSatisfaction,1200.0,2.72,1.09,1.0,2.0,3.0,4.0,4.0
EmpHourlyRate,1200.0,65.98,20.21,30.0,48.0,66.0,83.0,100.0
EmpJobInvolvement,1200.0,2.73,0.71,1.0,2.0,3.0,3.0,4.0
EmpJobLevel,1200.0,2.07,1.11,1.0,1.0,2.0,3.0,5.0
EmpJobSatisfaction,1200.0,2.73,1.10,1.0,2.0,3.0,4.0,4.0
NumCompaniesWorked,1200.0,2.66,2.47,0.0,1.0,2.0,4.0,9.0
EmpLastSalaryHikePercent,1200.0,15.22,3.63,11.0,12.0,14.0,18.0,25.0


In [5]:
categorical_columns = data.select_dtypes(include="object").columns.drop("EmpNumber")
for column in categorical_columns:
    print(column)
    print(data[column].value_counts().to_string())
    print()

Gender
Gender
Male      725
Female    475

EducationBackground
EducationBackground
Life Sciences       492
Medical             384
Marketing           137
Technical Degree    100
Other                66
Human Resources      21

MaritalStatus
MaritalStatus
Married     548
Single      384
Divorced    268

EmpDepartment
EmpDepartment
Sales                     373
Development               361
Research & Development    343
Human Resources            54
Finance                    49
Data Science               20

EmpJobRole
EmpJobRole
Sales Executive              270
Developer                    236
Manager R&D                   94
Research Scientist            77
Sales Representative          69
Laboratory Technician         64
Senior Developer              52
Manager                       51
Finance Manager               49
Human Resources               45
Technical Lead                38
Manufacturing Director        33
Healthcare Representative     33
Data Scientist                20
Re

## 3. Outliers and logical consistency

In [6]:
numeric_columns = data.select_dtypes(include="number").columns
rows = []
for column in numeric_columns:
    q1, q3 = data[column].quantile([0.25, 0.75])
    spread = q3 - q1
    count = int(((data[column] < q1 - 1.5 * spread) | (data[column] > q3 + 1.5 * spread)).sum())
    rows.append((column, count))
outliers = pd.DataFrame(rows, columns=["column", "outlier_count"]).sort_values("outlier_count", ascending=False)
outliers[outliers["outlier_count"] > 0]

,column,outlier_count
18,PerformanceRating,326
12,TrainingTimesLastYear,188
16,YearsSinceLastPromotion,88
14,ExperienceYearsAtThisCompany,56
11,TotalWorkExperienceInYears,51
8,NumCompaniesWorked,39
15,ExperienceYearsInCurrentRole,16
17,YearsWithCurrManager,11


In [7]:
consistency_checks = {
    "Years at this company greater than total experience": (data["ExperienceYearsAtThisCompany"] > data["TotalWorkExperienceInYears"]).sum(),
    "Years in current role greater than years at this company": (data["ExperienceYearsInCurrentRole"] > data["ExperienceYearsAtThisCompany"]).sum(),
    "Years since last promotion greater than years at this company": (data["YearsSinceLastPromotion"] > data["ExperienceYearsAtThisCompany"]).sum(),
    "Years with current manager greater than years at this company": (data["YearsWithCurrManager"] > data["ExperienceYearsAtThisCompany"]).sum(),
    "Total experience longer than working age (age minus 14)": (data["TotalWorkExperienceInYears"] > data["Age"] - 14).sum(),
}
pd.Series(consistency_checks, name="rows_failing_check")

Years at this company greater than total experience              0
Years in current role greater than years at this company         0
Years since last promotion greater than years at this company    0
Years with current manager greater than years at this company    0
Total experience longer than working age (age minus 14)          0
Name: rows_failing_check, dtype: int64

**Decision on outliers:** the values flagged above are genuine values (for example long experience or long time since promotion), not data entry mistakes, so they are **kept**. The tree based models used later are not sensitive to them, and the distance based models use scaling.

## 4. Transformations

In [8]:
label_maps = {
    "EmpEducationLevel": {1: "Below College", 2: "College", 3: "Bachelor", 4: "Master", 5: "Doctor"},
    "EmpEnvironmentSatisfaction": {1: "Low", 2: "Medium", 3: "High", 4: "Very High"},
    "EmpJobInvolvement": {1: "Low", 2: "Medium", 3: "High", 4: "Very High"},
    "EmpJobSatisfaction": {1: "Low", 2: "Medium", 3: "High", 4: "Very High"},
    "EmpRelationshipSatisfaction": {1: "Low", 2: "Medium", 3: "High", 4: "Very High"},
    "EmpWorkLifeBalance": {1: "Bad", 2: "Good", 3: "Better", 4: "Best"},
    "PerformanceRating": {1: "Low", 2: "Good", 3: "Excellent", 4: "Outstanding"},
}

# Modelling dataset: numeric codes for ordered scales, text for unordered categories
cleaned = data.copy()
cleaned["EmpNumber"] = cleaned["EmpNumber"].astype(str).str.strip()
for column in categorical_columns:
    cleaned[column] = cleaned[column].astype(str).str.strip()

# Reporting dataset: readable labels for ordered scales
labelled = cleaned.copy()
for column, mapping in label_maps.items():
    labelled[column] = labelled[column].map(mapping)

print("Labelled dataset missing values after mapping:", int(labelled.isna().sum().sum()))
cleaned.shape, labelled.shape

Labelled dataset missing values after mapping: 0


((1200, 28), (1200, 28))

**Transformation summary**

| Type of column | Treatment in the modelling stage |
|---|---|
| Ordered scales (satisfaction, education, work life balance, job level) | kept as numbers because the order carries meaning |
| Unordered categories (department, job role, education background, marital status, travel frequency) | converted to separate yes or no columns (one hot encoding) inside the model pipeline |
| Two value columns (gender, overtime, attrition) | converted to a single 0 or 1 column inside the model pipeline |
| Numeric measurements (age, hourly rate, experience years and so on) | standardised to mean 0 and standard deviation 1 for the models that need it |

The encoding is done **inside** the model pipeline (see `train_model.ipynb`) so exactly the same steps are applied to new employees when predicting.

## 5. Save processed data

In [9]:
PROCESSED.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(PROCESSED / "employee_cleaned.csv", index=False)
labelled.to_csv(PROCESSED / "employee_labelled.csv", index=False)
print("Saved:", sorted(p.name for p in PROCESSED.glob("*.csv")))

Saved: ['employee_cleaned.csv', 'employee_labelled.csv']
